# **Exercícios Aula 3**: Modelagem analítica (OLTP -> OLAP)

----

- Você recebeu um arquivo extraído de uma rede de supermercados com com 100.000 registros. 

- Transformar dados brutos em um Star Schema e validar o ganho de performance analítica.

- O .csv pode ser adquirido via: https://drive.google.com/file/d/1YBpZTsErQ2JBKgCs3dfD8QTc54JZf4-e/view?usp=drive_link

-----------------

## EXERCÍCIO 1: Modelando as Dimensões

- Crie um DataFrame chamado dim_produto contendo apenas as colunas únicas de produto e categoria.

- Crie uma coluna id_produto (pode ser o índice + 1).

- Crie um DataFrame chamado dim_loja contendo apenas a coluna única cidade_loja.

- Crie uma coluna id_loja.

- Garanta que os IDs sejam do tipo int32 (para gastar pouca memória).

- Imprima o .head() e o .info() de cada dimensão para verificar o resultado.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Carregar dados brutos
df_oltp = spark.read.csv("/Workspace/Users/matheus.rj25@hotmail.com/lab-ciencia-de-dados-senac/Atividades/data/olpt_mercado.csv", header=True, inferSchema=True)

# Criar dimensão produto
dim_produto = df_oltp.select("produto", "categoria").distinct()
dim_produto = dim_produto.withColumn("id_produto", (F.row_number().over(Window.orderBy("produto", "categoria"))).cast("int"))
dim_produto = dim_produto.select("id_produto", "produto", "categoria")
display(dim_produto)

# Criar dimensão loja
dim_loja = df_oltp.select("cidade_loja").distinct()
dim_loja = dim_loja.withColumn("id_loja", (F.row_number().over(Window.orderBy("cidade_loja"))).cast("int"))
dim_loja = dim_loja.select("id_loja", "cidade_loja")
display(dim_loja)

# Validar tipos
print(dim_produto.dtypes)
print(dim_loja.dtypes)

-----------------

## EXERCÍCIO 2: Construindo a Tabela Fato

- Crie a fato_vendas a partir do df_oltp.

- Substitua as colunas de texto (produto, categoria, cidade_loja) pelos seus respectivos IDs criados nas dimensões.

- Mantenha apenas as colunas: id_produto, id_loja, data_venda, valor_venda, quantidade.

- Converta as colunas de ID para float64.

- Imprima o .head() e o .info() para verificar se o resultado é numérico.

- Faça uma comparação de memória do original e o fato.

In [0]:
import pyspark.sql.functions as F

# Join df_oltp com dim_produto para obter id_produto
fato_vendas = df_oltp.join(dim_produto, on=["produto", "categoria"], how="left")

# Join com dim_loja para obter id_loja
fato_vendas = fato_vendas.join(dim_loja, on="cidade_loja", how="left")

# Selecionar apenas as colunas da tabela fato
fato_vendas = fato_vendas.select(
    "id_produto", "id_loja", "data_venda", "valor_venda", "quantidade"
)

# Converter colunas de ID para double (equivalente a float64)
fato_vendas = fato_vendas \
    .withColumn("id_produto", F.col("id_produto").cast("double")) \
    .withColumn("id_loja", F.col("id_loja").cast("double"))

print("=== fato_vendas .head() ===")
display(fato_vendas.limit(5))

print("=== fato_vendas .info() (printSchema) ===")
fato_vendas.printSchema()

# Comparação de memória (convertendo para pandas para medir)
df_oltp_pd = df_oltp.toPandas()
fato_pd = fato_vendas.toPandas()

mem_original = df_oltp_pd.memory_usage(deep=True).sum()
mem_fato = fato_pd.memory_usage(deep=True).sum()

print(f"\n=== Comparação de Memória ===")
print(f"df_oltp (original): {mem_original / 1024 / 1024:.2f} MB")
print(f"fato_vendas (star schema): {mem_fato / 1024 / 1024:.2f} MB")
print(f"Redução: {((mem_original - mem_fato) / mem_original) * 100:.2f}%")

-----------------

## EXERCÍCIO 3: Benchmark de Performance

- Calcule a soma de valor_venda por categoria usando o df_oltp original.

- Calcule a mesma soma usando o modelo Star Schema (Dica: Agrupe a fato_vendas pelo id_produto primeiro, depois faça o merge com a dimensão para pegar o nome da categoria).

- Utilize time.time() para medir o tempo de ambos.

- Calcule o ganho de performance.

In [0]:
import time
import pyspark.sql.functions as F

# 1. Consulta no OLTP original (df_oltp) — soma de valor_venda por categoria
start_oltp = time.time()
resultado_oltp = df_oltp.groupBy("categoria") \
    .agg(F.sum("valor_venda").alias("total_venda")) \
    .orderBy("categoria")
resultado_oltp.collect()  # força a execução
end_oltp = time.time()
tempo_oltp = end_oltp - start_oltp

# 2. Consulta no Star Schema (fato_vendas + dim_produto)
start_star = time.time()
fato_agg = fato_vendas.groupBy("id_produto") \
    .agg(F.sum("valor_venda").alias("total_venda"))
resultado_star = fato_agg.join(dim_produto, on="id_produto", how="left") \
    .groupBy("categoria") \
    .agg(F.sum("total_venda").alias("total_venda")) \
    .orderBy("categoria")
resultado_star.collect()  # força a execução
end_star = time.time()
tempo_star = end_star - start_star

print("=== Benchmark de Performance ===")
print(f"Tempo OLTP (original): {tempo_oltp:.4f}s")
print(f"Tempo Star Schema: {tempo_star:.4f}s")
print(f"Ganho de performance: {tempo_oltp / tempo_star:.2f}x")

print("\n=== Resultado OLTP ===")
display(resultado_oltp)

print("\n=== Resultado Star Schema ===")
display(resultado_star)

----------------